# 15 — Frozen agent rules (ratified 2026-07-30)

The constants declared in notebook 10 (marked `pending supervisor ratification` in its source) are
ratified **by the project owner on 2026-07-30**, recorded in `docs/supervisor_decision_sheet.md`.
The supervisor reviews the final draft; this page records who decided and when, so nothing
downstream presents these as externally signed off. From this date the values are frozen — any
change is a new, dated version — and the one-shot sealed-year evaluation (17) runs on exactly these.

**Triage of the externally pasted ratification text** (adopted only after correction — the numbers
were right, the labels were not):

- "Low/Medium/High pressure weights 0.2/0.6/0.2" → they are **component weights** on
  (>18-week share, >52-week share, G&A occupancy).
- "Five v1 action multipliers 0.70/0.40/1.20/1.05/0.80" → they are **two pressure thresholds, two
  band thresholds and one occupancy threshold**; nothing multiplies an action.
- Receiving rule "provider–specialty group with ≥ 30 validation rows" → **impossible on the real
  simulation** — the largest provider group has 18 origin months (asserted below) — so it would
  drive REDISTRIBUTE_IN to 0, the opposite of its stated goal. Rejected. The achievable form of the
  same intent is already arithmetic in the ratified thresholds: a within-ICB percentile rank ≤ 0.30
  requires an ICB of at least ceil(1/0.30) = 4 providers, so no smaller ICB can ever host a
  receiver — asserted below, not assumed.
- "Independent sector not eligible as receivers" → already true by construction: the simulation
  cohort is `fl_allocatable` (public providers only). Documented, no new code.

**The four frozen decisions** (questions from `docs/supervisor_decision_sheet.md`):

1. **Constants — ratified as declared** (correct semantics in the constants cell below).
2. **Receiving side — in-ICB only, minimum group 4 by arithmetic.** v1's REDISTRIBUTE_IN stays
   dormant (0 of 2,337, frozen as documented behaviour); the sender-side small-ICB pathology
   (a 1-provider ICB always ranks 1.0, so it can auto-classify OUT) is measured below and
   reported, not redesigned.
3. **Multi-seed check — approved**: seeds 42 / 123 / 999 declared before running, executed in
   `src/19_neural_c110_multiseed.ipynb`, every seed reported.
4. **Modelling stance — persistence remains the operational forecast layer everywhere; modelling
   outside C_110 is exploratory; the C_110 neural claim carries whatever the multi-seed check shows.**

Nothing in this notebook recomputes a forecast, band or pressure score: it re-derives both decision
columns from the **stored** component columns of `agent_decisions_h6_c110.csv` under the frozen
constants and asserts they reproduce notebook 10's stored decisions exactly — the freeze is proven,
not copied.

In [1]:
# Goal: prove the frozen constants reproduce the stored v1 and v2 decisions exactly (2,337 rows).
import pandas as pd                          # the stored decision table
import numpy as np                           # np.select re-derives the decision classes

# frozen constants - ratified 2026-07-30; identical values to notebook 10's declared block
W_GT18, W_GT52, W_OCC = 0.2, 0.6, 0.2        # pressure component weights (>18w, >52w, occupancy)
P_HIGH, P_MID = 0.70, 0.40                   # pressure thresholds (weighted within-month percentile)
G_RISK, G_STABLE, OCC_SLACK = 1.20, 1.05, 0.80   # band growth-risk / stability, occupancy slack
OUT_THR, IN_THR = 0.80, 0.30                 # v2 within-ICB sender / receiver rank thresholds

dec = pd.read_csv("../data/processed/agent_decisions_h6_c110.csv", parse_dates=["month"])
assert dec.shape == (2337, 17), dec.shape    # a regenerated decisions csv trips this

# v1 re-derived from the stored pressure score, band and occupancy columns - notebook 10's rule
high = dec["pressure_score"] >= P_HIGH
mid = (dec["pressure_score"] >= P_MID) & (dec["pressure_score"] < P_HIGH)
growth_risk = dec["upper_6m"] >= G_RISK * dec["total_waiting"]
stable = dec["upper_6m"] <= G_STABLE * dec["total_waiting"]
slack = dec["occupancy_ga"] < OCC_SLACK      # NaN occupancy compares False - unknown never counts as slack
v1 = np.select([high & growth_risk, mid & stable & slack],
               ["EXPAND_CAPACITY", "REDISTRIBUTE_IN"], "STATUS_QUO")
assert (v1 == dec["decision"]).all()         # any drift between frozen constants and 10's stored v1 trips this

# v2 re-derived from the stored within-ICB percentile ranks - notebook 10's rule
snd = (dec["icb_p_rank"] >= OUT_THR) & (dec["icb_o_rank"] >= OUT_THR)
rcv = (dec["icb_p_rank"] <= IN_THR) & (dec["icb_o_rank"] <= IN_THR)
v2 = np.select([snd, rcv], ["REDISTRIBUTE_OUT", "REDISTRIBUTE_IN"], "STATUS_QUO")
assert (v2 == dec["decision_v2"]).all()      # same guarantee for the v2 column

assert dec["decision"].value_counts().to_dict() == {"STATUS_QUO": 2003, "EXPAND_CAPACITY": 334}
# v1 REDISTRIBUTE_IN is absent because it selects zero rows - the documented dormant class
assert dec["decision_v2"].value_counts().to_dict() == {
    "STATUS_QUO": 1775, "REDISTRIBUTE_OUT": 457, "REDISTRIBUTE_IN": 105}
print("frozen constants reproduce 10's stored decisions exactly:",
      f"v1 {dec['decision'].value_counts().to_dict()} | v2 {dec['decision_v2'].value_counts().to_dict()}",
      f"| {dec['provider_code'].nunique()} providers, {dec['month'].nunique()} origin months")

frozen constants reproduce 10's stored decisions exactly: v1 {'STATUS_QUO': 2003, 'EXPAND_CAPACITY': 334} | v2 {'STATUS_QUO': 1775, 'REDISTRIBUTE_OUT': 457, 'REDISTRIBUTE_IN': 105} | 132 providers, 18 origin months


In [2]:
# Goal: the frozen receiving-side rule, proven from the stored decisions - and the pasted N=30
# rule shown impossible on the same data.
rows_per_provider = dec.groupby("provider_code").size()
assert rows_per_provider.max() == 18         # the simulation has 18 origin months - no provider group
# can ever reach 30 rows, which is why the pasted minimum-group rule was rejected

icb_n = dec.groupby("icb_code")["provider_code"].nunique().rename("icb_size")
dec = dec.merge(icb_n, on="icb_code")

# the arithmetic minimum group: rank(pct=True) gives a lowest rank of 1/n, and 1/n <= 0.30 needs
# n >= 4, so the frozen IN_THR already forbids receivers in ICBs smaller than 4 providers
assert int(np.ceil(1 / IN_THR)) == 4
assert dec.loc[dec["decision_v2"] == "REDISTRIBUTE_IN", "icb_size"].min() >= 4
# would trip if any stored receiver sat in a tiny ICB - the rule is emergent, not decorative

# sender side, measured not redesigned: the max rank is always 1.0, so OUT can fire in any ICB size
out = dec[dec["decision_v2"] == "REDISTRIBUTE_OUT"]
small_out = out[out["icb_size"] < 4]
print(f"OUT rows in ICBs with < 4 providers (no local receiver is ever possible there): "
      f"{len(small_out)} of {len(out)} = {len(small_out) / len(out):.1%}")
print(f"ICBs with < 4 cohort providers: {(icb_n < 4).sum()} of {len(icb_n)}")

# the receiver-match share must be unchanged by the freeze - it is the same rule, proven above
slots = dec[dec["decision_v2"] == "REDISTRIBUTE_IN"][["month", "icb_code"]].drop_duplicates()
m = out.merge(slots, on=["month", "icb_code"], how="left", indicator=True)
share = (m["_merge"] == "both").mean()
assert round(share, 3) == 0.136              # the documented 13.6% of 457 OUT provider-months
print(f"OUT months with an in-ICB receiver slot: {share:.1%} of {len(out)}")

OUT rows in ICBs with < 4 providers (no local receiver is ever possible there): 246 of 457 = 53.8%
ICBs with < 4 cohort providers: 21 of 36
OUT months with an in-ICB receiver slot: 13.6% of 457


In [3]:
# Goal: agent_frozen_summary.csv - the frozen-rules outcome table 17 will be compared against.
rows = []
for rule, col in [("v1", "decision"), ("v2", "decision_v2")]:
    for cls, grp in dec.groupby(col):
        rows.append({"rule": rule, "decision": cls, "n": len(grp),
                     "share_of_2337": round(len(grp) / len(dec), 4),
                     "median_pressure": grp["pressure_score"].median(),
                     "median_realized_change": grp["realized_change"].median()})
summary = pd.DataFrame(rows)
summary.to_csv("../data/processed/agent_frozen_summary.csv", index=False)
print(summary.round(3).to_string(index=False))
print("-> data/processed/agent_frozen_summary.csv (frozen 2026-07-30; 17 evaluates exactly these rules)")

rule         decision    n  share_of_2337  median_pressure  median_realized_change
  v1  EXPAND_CAPACITY  334          0.143            0.794                     0.5
  v1       STATUS_QUO 2003          0.857            0.451                    37.0
  v2  REDISTRIBUTE_IN  105          0.045            0.183                     5.0
  v2 REDISTRIBUTE_OUT  457          0.196            0.723                    81.0
  v2       STATUS_QUO 1775          0.760            0.471                    14.0
-> data/processed/agent_frozen_summary.csv (frozen 2026-07-30; 17 evaluates exactly these rules)


## What the freeze proves

- The frozen constants reproduce notebook 10's stored decisions **bit for bit** on all 2,337
  provider-months (both rule sets asserted equal) — the freeze is a verified restatement, not a
  re-implementation that could drift.
- The receiving-side minimum group (4 providers per ICB) is **enforced by the ratified 0.30
  threshold itself**; every stored receiver row already satisfies it (asserted).
- The sender-side gap is now quantified (cell above): OUT months in sub-4-provider ICBs can never
  find a local receiver under the frozen in-ICB rule — that share of OUT months is an inherent
  property of the frozen rule, carried into 17's evaluation and reported in the write-up, not
  patched here.
- Next and last step: `17_sealed_year_eval.ipynb`, written and run **exactly once** on explicit
  instruction — persistence + conformal bands, the linear and neural FL models, and the v2 rule
  above on FY 2025-26, reported as-is.